# Day 6 | ILT 2: Grain Definition & Dimension Design
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 2 hours &nbsp;|&nbsp; **Level:** Intermediate &nbsp;|&nbsp; **Tags:** grain, dimensional-modeling, surrogate-key, conformed-dimension, scd1, scd2

---
**Goal:** Learn Kimball's #1 rule — declare the grain before building anything — then design every one of `fact_sales`'s 6 dimensions, using GlobalMart's real Silver tables as the source.

---
**INSTRUCTOR NOTE:**
This is the longest session of Day 6 for a reason: grain and dimension design are the decisions that are hardest to undo later. Getting the grain wrong means rebuilding the fact table from scratch. This session is still conceptual (no runnable code) — but treat it as a design document students will follow almost line-by-line in this afternoon's Hands-On.

## Learning Objectives

By the end of this session, students will be able to:

1. Define **grain** and explain why it must be declared before any table is built
2. Justify why `fact_sales`'s grain is "one row per order line item" over coarser or finer alternatives
3. Design each of GlobalMart's 6 dimension tables — source table, natural key, attributes, surrogate key
4. Explain why Gold **reuses** the surrogate key Silver already generated for SCD2 dimensions, but **generates a fresh one** (via `sha2()`) for the rest
5. Explain the SCD Type 1 vs. Type 2 decision framework, and recognize which of the 6 dimensions already have it applied (from Day 5) versus which don't need it
6. Explain what a **conformed dimension** is and why it matters at GlobalMart's scale

---
## Section 1: What Is Grain?

**INSTRUCTOR NOTE:**
Ask: *"If I say `fact_sales` has 500,000 rows, what does one row represent?"* If nobody can answer precisely, that's the point of this section — grain is exactly that answer, stated before a single line of code is written.

---

**Grain** is the single, unambiguous statement of what one row in a fact table represents. Kimball's rule: **declare the grain in one sentence, before designing a single column.** Every measure and every dimension you add afterward must be true *at that grain* — if it isn't, either the grain is wrong or the column doesn't belong in this table.

### GlobalMart's declared grain

> **One row in `fact_sales` = one line item within one order** — i.e., one row per `(order_id, product_id)` combination in `order_items`.

### A worked example

Order `OR-000001` has two line items in `order_items`:

| order_item_id | order_id | product_id | quantity |
|---|---|---|---|
| OR-J-283399 | OR-094494 | PRD-00410 | 1 |
| OR-KSC-283400 | OR-094494 | PRD-00107 | 5 |

At the chosen grain, this order contributes **exactly 2 rows** to `fact_sales` — one per product. `Actual_price`/`Discounted_price` are looked up per row from `silver.products`, and `Sales_amount` is computed from them during Day 7's fact build, so each row carries its own correct revenue number.

### Why not a coarser grain — "one row per order"?

| If grain were "one row per order"... | Problem |
|---|---|
| Order `OR-094494`'s 2 products collapse into 1 row | You lose per-product `Quantity_purchased` and `Sales_amount` — you can no longer answer "which product sold best?" |
| `product_id` would need to become a list/array in one cell | Breaks the entire star schema pattern — dimensions expect one clean foreign key per fact row |

**Coarser grain always loses information you cannot get back later without re-processing Silver.** This is the single biggest reason to get grain right the first time.

### Why not a finer grain — is there anything finer than a line item?

Not for GlobalMart's current sources. A finer grain would exist if, say, a single line item could be split across multiple shipments or fulfilled from multiple warehouse locations — GlobalMart's `order_items` table doesn't track that, so line item is the finest grain the source data actually supports. If GlobalMart later added shipment-level tracking, that would justify a **new**, finer-grained fact table (e.g. `fact_shipments`) — not a change to `fact_sales`.

### A different business process entirely

A return/refund event (from `returns.csv`) is a genuinely different business process — it would become its own `fact_returns` table one day, at its own grain, not a variant of `fact_sales`. Not built in this course, but worth naming so "grain" and "business process" don't get confused with each other.

### The grain test

Before adding any column to a fact table, ask: *"Is this value true for exactly one line item, or could it vary within the order?"* `Quantity_purchased` and `Discounted_price` pass (they're specific to one product in the order). `order_date` also passes (every line item in an order shares the same order date, and reaches `fact_sales` indirectly through `dim_date`). Something like "total number of items in this order" would **fail** — that's an order-level aggregate, not a line-item-level fact. It doesn't belong as a new column in `fact_sales` at all; if GlobalMart needed it, it would be computed on demand with `SUM(Quantity_purchased) GROUP BY Order_ID`, not stored.

### Consequence of this grain on the 4 measures

Because the grain is exactly one order line item, rolling `fact_sales` up to any level (customer, category, day, region) by summing `Quantity_purchased` or `Sales_amount` always produces a correct number — every row is a clean, non-overlapping slice of a sale. `Actual_price`/`Discounted_price` behave differently even at this same grain — summing *either* is never meaningful, because they're rates, not amounts. ILT 3 (next) gives this its full treatment: which measures are additive, and which aren't.</cell id="cell-grain-01">


---
## Section 2: Designing `dim_customer`

**Source:** `<your-catalog>.silver.customers` &nbsp;|&nbsp; **Natural key:** `customer_id` &nbsp;|&nbsp; **Grain:** one row per customer *version*

| Column | Why it's here |
|---|---|
| `customer_sk` | Surrogate key — **reused from Silver**, not generated in the Gold build. The join target every fact table uses. |
| `customer_id` | Natural/business key — traces back to the source system (`CustomerID` in `customers.csv`) |
| `full_name` | What the business actually looks at |
| `email`, `phone_number` | Contact attributes — also the fields most likely to change over a customer's lifetime |
| `is_current` | Marks which version of this customer is true *right now* |
| `effective_start_date`, `effective_end_date` | The date range this version of the customer was/is true for |

**INSTRUCTOR NOTE:**
This is the one place this session corrects a common assumption: SCD Type 2 for `dim_customer` is **not** a future topic. Day 5's Hands-On already built `silver.customers` as a full SCD2 table — it generated `customer_sk` and stamped every row with `is_current`/`effective_start_date`/`effective_end_date`. This afternoon's Gold build does not re-decide any of that; it selects these columns as-is and republishes them with `mode("overwrite")`. The `MERGE`-based mechanics for *updating* this table incrementally when a customer's details change — closing out the old version, inserting the new one — is a later session (Incremental Loading & SCD). Today, the SCD2 structure already exists; you're just moving it into Gold.

---
## Section 3: Designing `dim_product`

**Source:** `<your-catalog>.silver.products` &nbsp;|&nbsp; **Natural key:** `product_id` &nbsp;|&nbsp; **Grain:** one row per product *version*

| Column | Why it's here |
|---|---|
| `product_sk` | Surrogate key — **reused from Silver**, same reasoning as `dim_customer` |
| `product_id` | Natural/business key (e.g. `PRD-00001`) |
| `product_name` | Human-readable label for reports and Genie answers |
| `category`, `sub_category` | What "revenue by category" — one of the most common business questions — groups and filters by |
| `discounted_price_inr` | The attribute that actually changes and is the reason this dimension is SCD Type 2 — price history is the textbook use case |
| `is_current`, `effective_start_date`, `effective_end_date` | SCD2 control columns, same role and same source (Silver) as in `dim_customer` |

**Design note:** `discounted_price_inr` is exactly the kind of attribute that makes `dim_product` a realistic SCD Type 2 candidate — if GlobalMart runs a flash sale and drops a product's price, you generally want historical sales to keep reporting against the price that was true *at the time of the sale*, not be silently rewritten to today's price. Like `dim_customer`, this decision was already made and built in Day 5's Silver layer — Gold just carries it forward. `is_current` matters more than it might look here: Day 7's Gold reporting views filter on it directly (`WHERE p.is_current = true`) whenever they join `dim_product`.

---
## Section 4: Designing `dim_date`

**Source:** none — **generated**, not read from Silver &nbsp;|&nbsp; **Natural key:** the calendar date itself &nbsp;|&nbsp; **Grain:** one row per calendar day

Every other dimension has a Silver source table. `dim_date` never does, in any dimensional model — and that's deliberate: no source system ships a "table of dates." You generate the full calendar yourself, once.

**Sized to the data, not an arbitrary range.** Rather than hardcoding a wide guess (e.g. "2010 to 2027"), pull the real bounds straight from `silver.orders` — earliest `order_date`, latest `actual_delivery_date` — and generate exactly that range. This covers every date `fact_sales` will ever need to look up, with no wasted rows either side, and it means the spine automatically stays correct if GlobalMart's real order history grows.

| Column | Why it's here |
|---|---|
| `date_key` | Surrogate key — but see the callout below, this is the **one deliberate exception** to the reuse-or-hash convention |
| `date` | The actual calendar date, for display and date-math in BI tools |
| `year`, `quarter`, `month` | Pre-computed calendar hierarchy — avoids every query recomputing `QUARTER(date)` itself |
| `day_of_week` | Day-of-week name, for weekday/weekend and seasonality analysis |
| `is_weekend` | Common business flag — "do we sell more on weekends?" without date-math in every query |

> **Why `date_key` is an integer `YYYYMMDD`, not a hash:** every other freshly-generated surrogate key in this model is `sha2(natural_key, 256)` (Section 7). `dim_date` breaks that pattern on purpose — its surrogate key is a plain integer like `20260703`. This is a long-standing Kimball convention: it's human-readable in ad-hoc queries, it sorts correctly as a number, and it partitions Delta tables cleanly. Know this as the deliberate exception, not an inconsistency.

**INSTRUCTOR NOTE:**
`dim_date` is the textbook example of a **conformed dimension** — see Section 8. Every fact table GlobalMart ever builds (`fact_sales`, a future `fact_returns`, etc.) joins to this exact same table. It is built once and reused forever.

---
## Section 5: Designing `dim_address` and `dim_payment_method`

### `dim_address`

**Source:** `<your-catalog>.silver.address` (singular) &nbsp;|&nbsp; **Natural key:** `address_id` &nbsp;|&nbsp; **Grain:** one row per address record

| Column | Why it's here |
|---|---|
| `address_sk` | Surrogate key — generated fresh here in Gold, since Silver never assigned this table one |
| `address_id` | Natural/business key, traces back to source |
| `customer_id` | Lets you filter "this customer's addresses" without going through the fact table |
| `city`, `state`, `pincode` | The entire reason this dimension exists — "revenue by city/state" reporting |
| `address_type` | Distinguishes `Billing` vs. `Shipping` |

> **The many-to-many problem — and how it's actually resolved.** One customer can have *multiple* addresses (a Billing address and a Shipping address, at minimum) — a plain dimension table can't say by itself "which address did *this specific order line* ship to." That's a real problem, but it is **not** solved with a bridge table. Day 7's `fact_sales` build resolves it with a window-function ranking: rank each customer's addresses (prefer `Shipping`), keep the top-ranked one, join on `customer_id`. `dim_address` itself stays a plain, honest dimension — the resolution logic lives entirely in the fact build. (Day 7 HOL 1 does build one illustrative bridge table, for a hypothetical products↔campaigns relationship, purely to teach the bridge-table pattern in general — that example has nothing to do with addresses.)

### `dim_payment_method`

**Source:** `<your-catalog>.silver.payment_methods` (the small lookup table) &nbsp;|&nbsp; **Natural key:** `payment_method_id` &nbsp;|&nbsp; **Grain:** one row per payment method (~5 rows total)

| Column | Why it's here |
|---|---|
| `payment_method_sk` | Surrogate key, generated fresh here in Gold |
| `payment_method_id` | Natural/business key (e.g. `PM-002`) |
| `method_name` | The human-readable value ("UPI", "Credit Card") — `PM-002` means nothing to a business user on its own |

> **Common mistake — don't confuse the two payment tables.** `dim_payment_method` sources from `payment_methods` (the 5-row lookup: `PaymentMethodID`, `MethodName`), **not** from `payments` (the transactional table recording every individual payment event per order — gift card usage, coupon amount, payment date). `payments` is transactional detail that belongs near the fact, not a slowly-changing dimension. A tiny reference table like `dim_payment_method` is sometimes called a **mini-dimension** — worth knowing the term.

Both of these dimensions get a surrogate key generated fresh in Gold, even though they're small and simple — keeping every dimension's join shape identical means `fact_sales` never needs special-case join logic per dimension (a rule Day 7 will show one deliberate exception to — see ILT 3).

---
## Section 6: Designing `dim_orders`

**Source:** `<your-catalog>.silver.orders` &nbsp;|&nbsp; **Natural key:** `order_id` &nbsp;|&nbsp; **Grain:** one row per order

| Column | Why it's here |
|---|---|
| `order_sk` | Surrogate key, generated fresh here in Gold — same uniform pattern as `dim_address`/`dim_payment_method` |
| `order_id` | Natural/business key, traces back to source |
| `customer_id` | Lets you filter "this customer's orders" without going through the fact table |
| `order_date` | Order header date — also independently reachable via `dim_date` through the fact, kept here too since it's a natural attribute of the order itself |
| `shipping_tier_id`, `supplier_id` | Carried as plain IDs — no backing lookup table exists for either, so they stay undecoded here |
| `order_channel` | The one attribute with real reporting value — "Online" vs. "Retail PoS" |

> **This is the sixth dimension — it didn't exist in earlier drafts of this course's spec, and it's easy to assume `order_id` has no dimension at all.** It does: `dim_orders` is real, and today's Hands-On builds it. What's still true is that `fact_sales` (Day 7) doesn't join to it by surrogate key — it carries `Order_ID` directly as a natural key instead, the same deliberate simplification the fact table applies everywhere else. `dim_orders` exists as its own queryable Gold table regardless; ILT 3 (next) revisits this exact point under "degenerate dimensions."

---
## Section 7: Surrogate Keys — the Kimball Standard, and Where GlobalMart Takes a Shortcut

**Natural key** (a.k.a. business key): the identifier the *source system* assigns — `customer_id`, `product_id`, `PM-002`. It means something to the business and to the source application.

**Surrogate key**: an identifier *Gold* owns, meaningless outside the warehouse, whose only job is to be a stable, efficient join target. The Kimball-standard rule, and cert material: **dimension tables get a surrogate key, and fact tables join to dimensions on that surrogate key — never the natural key.**

### Two ways a dimension gets its surrogate key in this model

| | Reuse from Silver | Generate fresh in Gold |
|---|---|---|
| **Applies to** | `dim_customer`, `dim_product` | `dim_address`, `dim_payment_method`, `dim_orders` |
| **Why** | Silver already built these as SCD2 tables (Day 5) and generated the key there — regenerating it in Gold would just recompute the same thing, or worse, silently collapse SCD2 history if done carelessly | Silver never assigned these a surrogate key, so Gold generates one: `sha2(natural_key, 256)` |
| **Method** | `SELECT` the existing `customer_sk` / `product_sk` column as-is | `address_sk = sha2(address_id, 256)`, and the same pattern for the other two |

(`dim_date`'s `date_key` is the one further exception to *both* paths — a plain `YYYYMMDD` integer, Section 4.)

### Why a hash-based surrogate key at all

| Reason | Explanation |
|---|---|
| **Idempotent** | Re-running the Gold build produces the *exact same* key for the same natural key, every time. An auto-incrementing counter would not. |
| **Decouples from source-system key changes** | If Supabase or an ADLS file drop ever changes its ID format, `fact_sales` doesn't care — Gold's keys are Gold-owned. |
| **Stable under SCD Type 2** | When a dimension versions history, the fact must be able to point at the exact dimension row version that was true *at the moment of the sale*. Only a key Gold controls (or inherits from Silver's own SCD2 build) can support that — a bare natural key can't distinguish between two historical versions of the same customer. |
| **Join performance** | A fixed-length hash joins faster than long, variable-length string natural keys at `fact_sales`'s scale. |

### The shortcut GlobalMart's real `fact_sales` actually takes

Here's the honest part: even though every dimension above has a proper surrogate key, **`fact_sales` itself (built Day 7) does not use any of them.** It carries `Customer_ID`, `Product_ID`, `Order_ID`, `Address_ID`, `Payment_ID`, and `Time_ID`. Five of these six join their dimension directly by natural key — `Customer_ID`→`dim_customer`, `Product_ID`→`dim_product`, `Order_ID`→`dim_orders`, `Address_ID`→`dim_address`, `Time_ID`→`dim_date` (which happens to *be* `dim_date.date_key`, the one dimension where natural and surrogate collapse into the same column). **`Payment_ID` is the exception to the exception:** it comes from `silver.payments` (the transactional payment record), a completely different ID space from `dim_payment_method.payment_method_id` — so `fact_sales` doesn't resolve to `dim_payment_method` through any column at all. ILT 3 Section 4 covers this in full.

This natural-key choice is a **deliberate, documented simplification** for this training build, not an oversight — and it has a real cost: no point-in-time SCD2 accuracy. If a customer's email changes, every one of their past `fact_sales` rows now resolves to the *current* `dim_customer` row when joined, not the version that was true when each sale happened. You now know exactly what that costs, and why the surrogate-key discipline above is still the correct default to teach and default to build, even though this particular fact table takes the shortcut.</cell id="cell-surrogate-keys-06">


---
## Section 8: Conformed Dimensions

**INSTRUCTOR NOTE:**
This is a forward-looking concept — GlobalMart only has one fact table today, so conformance isn't tested yet. But it's the reason dimensions are worth designing carefully now rather than per-report later.

---

A **conformed dimension** is a dimension built once, with one definition, and reused unchanged by every fact table in the warehouse.

**GlobalMart's concrete example:** `dim_product` and `dim_customer`, built today, are designed to be reusable. If GlobalMart later builds `fact_returns` (from `returns.csv` — a real file that already exists in the raw data, just not built into a fact table in this course), that fact table would join to the *exact same* `dim_product` and `dim_customer` rows and surrogate keys already sitting in `<your-catalog>.gold`. That's what makes "revenue by category" (from `fact_sales`) and "return rate by category" (from a future `fact_returns`) automatically agree — both queries filter through the identical `dim_product.category` values, because both fact tables conform to the same dimension. `dim_date` is the same story: any future fact table joins to today's exact `dim_date` table, not a re-derived one.

**Why this matters:** without conformance, two teams building two fact tables might each build their own "product dimension" with slightly different category groupings — and now "revenue by category" and "return rate by category" silently disagree, because they're not even using the same definition of "category." This is precisely the trust problem GlobalMart started this whole course trying to solve (Day 1, Problem #1 — Data Silos, and Problem #5 — Unclear Origin & Redundancy).

---
## Section 9: SCD Type 1 vs. Type 2 — Conceptual Preview

**INSTRUCTOR NOTE:**
Keep this section conceptual, as a decision *framework* — not an implementation lesson. The actual `MERGE`-based mechanics for SCD Type 1 and Type 2 are taught in a later session. Today's Hands-On builds every dimension as a simple current-state table (`mode("overwrite")`, one row per natural key) — no `is_current` or `effective_date` columns yet, except where Silver already built them (`dim_customer`, `dim_product`). Be explicit about that boundary so students don't expect SCD columns everywhere in this afternoon's lab and then feel like something is "missing."

---

Source data changes over time — a customer updates their email, GlobalMart adjusts a product's price. **Slowly Changing Dimension (SCD)** handling is the decision about what happens to the *old* value when that happens.

| | SCD Type 1 | SCD Type 2 |
|---|---|---|
| **Behavior** | Overwrite the old value in place | Keep the old row, insert a new row, mark which is current |
| **History preserved?** | No — only the current value survives | Yes — every version, with effective date ranges |
| **Extra columns needed** | None | `is_current`, `effective_start_date`, `effective_end_date`, a versioned surrogate key |
| **GlobalMart example** | Fixing a typo'd `city` on an address; a payment method's display name changing | A customer's `email`/city changes and you still need to know which one was true when a *past* order shipped; a product's `discounted_price_inr` changes and past revenue must still reflect the price paid at the time |

### The decision rule

Ask: *"Does anyone need to know what this looked like in the past?"* No → Type 1 (simpler, overwrite). Yes → Type 2 (preserve history).

### Applying it to GlobalMart's 6 dimensions

| Dimension | SCD type | Why |
|---|---|---|
| `dim_customer` | **Type 2 — already built this way** | Silver (Day 5) already generates `is_current`/`effective_start_date`/`effective_end_date`; email/phone changes and historical orders should still show what was true when the sale happened |
| `dim_product` | **Type 2 — already built this way** | Same Silver-built SCD2 columns; price history — see Section 3's flash-sale example |
| `dim_address` | Type 1 is often enough | Typo corrections don't usually need history; genuinely new addresses are just new rows (natural key `address_id` already handles that) |
| `dim_payment_method` | Type 1 | A ~5-row lookup table; a name correction doesn't need history |
| `dim_orders` | Type 1 | Order header attributes don't change after the order is placed |
| `dim_date` | Neither — static | Generated once, never "changes" in the SCD sense |

This table is a preview of a decision framework, not a build instruction for every dimension — `dim_customer` and `dim_product` already ship today with Type 2 behavior (inherited from Silver), while `dim_address`/`dim_payment_method`/`dim_orders` ship today as simple current-state tables. The calendar's later sessions on incremental loading and `MERGE` come back to implement Type 2 mechanics hands-on, for the dimensions that need it.</cell id="cell-scd-08">


---
## Session Summary

| Dimension | Source | Natural key | Surrogate key | Grain |
|---|---|---|---|---|
| `dim_customer` | `<your-catalog>.silver.customers` | `customer_id` | Reused from Silver — `customer_sk` (SCD2) | One row per customer version |
| `dim_product` | `<your-catalog>.silver.products` | `product_id` | Reused from Silver — `product_sk` (SCD2) | One row per product version |
| `dim_date` | Generated date spine | calendar date | `date_key` — `YYYYMMDD` integer (the one exception) | One row per calendar day |
| `dim_address` | `<your-catalog>.silver.address` | `address_id` | Fresh in Gold — `sha2(address_id, 256)` | One row per address |
| `dim_payment_method` | `<your-catalog>.silver.payment_methods` | `payment_method_id` | Fresh in Gold — `sha2(payment_method_id, 256)` | One row per payment method (~5 rows) |
| `dim_orders` | `<your-catalog>.silver.orders` | `order_id` | Fresh in Gold — `sha2(order_id, 256)` | One row per order |

| Topic | Key Takeaway |
|---|---|
| **Grain** | Declare it in one sentence before designing anything. `fact_sales` = one row per order line item. |
| **Coarser grain risk** | Loses per-product measures, breaks the star schema pattern |
| **Surrogate keys** | `dim_customer`/`dim_product` reuse the SCD2 key Silver already built; `dim_address`/`dim_payment_method`/`dim_orders` generate a fresh `sha2(natural_key)` in Gold; `dim_date` is the one further exception (`YYYYMMDD` integer) |
| **The fact table's shortcut** | Despite every dimension having a proper surrogate key, `fact_sales` (Day 7) joins all 6 by natural key instead — a deliberate simplification, at the cost of point-in-time SCD2 accuracy |
| **Conformed dimensions** | One definition, reused by every fact table — `dim_product`/`dim_customer` are built to be reused by a future `fact_returns` |
| **SCD Type 1 vs. Type 2** | `dim_customer`/`dim_product` already ship as Type 2 (from Silver); the rest ship as simple current-state tables today |

**Next:** Day 6 ILT 3 — Fact Tables Deep Dive, where these 6 dimensions meet `fact_sales`'s actual measures, additivity rules, and key design. Then this afternoon's Hands-On builds exactly what's designed here.</cell id="cell-summary-09">
